In [ ]:
import numpy as np
import pandas as pd
import sys
import os
import pickle
from collections import defaultdict
import json
sys.path.append("../../src/experiment_util") 
import experiment_utils
import experiment_constants
sys.path.append("../../src/irl") 
import data_corruption_utils
import irl_utils

In [ ]:
def simulate_policy(policy, tp, n, initial_state=None, n_states=12):
    """
    policy: array shape (num_actions, num_states)
    transition_matrix: array shape (num_states, num_actions, num_states)
    n: trajectory length
    start_state: initial state index
    """
    num_states, num_actions = policy.shape

    if initial_state is None:
        initial_state = np.random.randint(0, 12)

    states = [initial_state]
    actions = []

    current_state = initial_state
    for step in range(n):
        # sample action according to policy for current state
        try:
            action = np.random.choice(num_actions, p=policy[current_state, :])
        except ValueError as e:
            print(f"Policy probs do not sum to 1 at state {current_state}, step {step}. Sum={policy[:, current_state].sum()}, values={policy[:, current_state]}")
            raise

        actions.append(action)

        # sample next state from transition probabilities
        try:
            next_state = np.random.choice(num_states, p=tp[current_state, action])
        except ValueError as e:
            print(f"TP probs do not sum to 1 at current_state={current_state}, action={action}, step={step}. Sum={tp[current_state, action].sum()}, values={tp[current_state, action]}")
            raise

        states.append(next_state)
        current_state = next_state
        
    # ignore final next state
    return np.column_stack((states[:-1], actions))

def sample_content(traj, user_content_df, user):
    if user_content_df.empty:
        return np.zeros(1024)

    expected_dim = 1024

    # check embeddings for consistency and print full bad rows
    for idx, e in user_content_df["embedding"].items():
        arr = np.array(e)
        if arr.shape != (expected_dim,):
            print(f"[User: {user}] Bad embedding at row {idx}, shape={arr.shape}")
            print(user_content_df.loc[idx], "\n")

    # action remapping to actions without agreement
    action_map = np.array([0, 1, 2, 3, 3, 3])
    traj_mapped = traj.copy()
    traj_mapped[:, 1] = action_map[traj[:, 1].astype(int)]

    actions, action_counts = np.unique(traj_mapped[:, 1], return_counts=True)

    # group once instead of filtering repeatedly
    grouped = {a: df for a, df in user_content_df.groupby("action")}

    sampled_content = []
    for a, count in zip(actions, action_counts):
        if a == 0:  # skip wait actions
            continue
        if a in grouped:
            sampled = grouped[a].sample(n=count, replace=True).embedding.values
            sampled_content.extend(sampled)

    if not sampled_content:
        return np.zeros(expected_dim)

    stacked = np.vstack(sampled_content)
    return stacked.mean(axis=0)

def simulate_mixed_policy(policy1, policy2, tp1, tp2, n, frac=0.5, initial_state=None, n_states=12):
    """
    policy1, policy2:         arrays shaped (num_states, num_actions)
    tp1, tp2:                 arrays shaped (num_states, num_actions, num_states)
    n:                        trajectory length
    frac:                     fraction of trajectory generated by policy1/tp1
    initial_state:            state to start from; if None, uniform random
    """

    num_states, num_actions = policy1.shape
    assert policy2.shape == (num_states, num_actions), "policies must be same shape"
    assert tp1.shape == (num_states, num_actions, num_states), "tp1 wrong shape"
    assert tp2.shape == tp1.shape, "tp2 must match tp1 shape"

    # how many steps to run under policy1 / tp1
    split_point = int(frac * n)

    if initial_state is None:
        initial_state = np.random.randint(0, n_states)

    states = [initial_state]
    actions = []
    current_state = initial_state

    for step in range(n):
        # choose which policy / tp to use
        if step < split_point:
            policy = policy1
            tp = tp1
        else:
            policy = policy2
            tp = tp2

        try:
            action = np.random.choice(num_actions, p=policy[current_state, :])
        except ValueError:
            print(f"Policy probs do not sum to 1 at state {current_state}, step {step}. Sum={policy[current_state, :].sum()}, values={policy[current_state, :]}")
            raise

        actions.append(action)

        try:
            next_state = np.random.choice(num_states, p=tp[current_state, action])
        except ValueError:
            print(f"TP probs do not sum to 1 at current_state={current_state}, action={action}, step={step}. Sum={tp[current_state, action].sum()}, values={tp[current_state, action]}")
            raise

        states.append(next_state)
        current_state = next_state

    return np.column_stack((states[:-1], actions))


In [ ]:
with open("./config.json") as f:
    config = json.load(f)
    root_output_dir_path = config["root_output_dir_path"]


In [ ]:
# load content from posts and submissions
post_embedding_df = pd.read_pickle(root_output_dir_path + "/text_embedding.pkl")
submission_embedding_df = pd.read_pickle(root_output_dir_path + "/submission_text_embedding.pkl")
text_embedding_df = pd.concat([post_embedding_df, submission_embedding_df], ignore_index=True)

In [ ]:
sampled_matched_perturbed_df = pd.read_pickle(root_output_dir_path + "/sampled_matched_perturbed_df_w_gail_opt.pkl")

df = sampled_matched_perturbed_df
dropped_percent = 0
for run in df.run.unique():
    mask_left  = (df.run == run) & (df.perturb_percent == dropped_percent) & (df.russian == 1)
    mask_right = (df.run == (run + 10) % 25) & (df.perturb_percent == dropped_percent) & (df.russian == 0)

    df.loc[mask_left, "policy_nt"] = df.loc[mask_right, "gail_policy"].values
    df.loc[mask_left, "user_nt"] = df.loc[mask_right, "user"].values
    df.loc[mask_left, "tp_nt"] = df.loc[mask_right, "tp_perturbed_legalised"].values
# dfs for content based on troll status
troll_user_content_df = text_embedding_df[text_embedding_df.screen_name.isin(df[df.russian == 1].user.unique())].copy()
organic_user_content_df = text_embedding_df[text_embedding_df.screen_name.isin(df[df.russian == 0].user.unique())].copy()

In [ ]:
num_runs = 25
n=100
dropped_percent = 0.0

troll_generated = []
organic_generated = []

df = sampled_matched_perturbed_df        
for percentage_non_troll in [0.1,0.2,0.3,0.4,0.5,0.6]:
    for run in range(num_runs):
        
        print(dropped_percent,"run:",run)
        seed = 100 + run

        df_troll_sampled = df[(df.run == run) & (df.perturb_percent == dropped_percent) & (df.russian == 1)].copy()
        df_organic_sampled = df[(df.run == run) & (df.perturb_percent == dropped_percent) & (df.russian == 0)].copy()
        # generate policies
        df_troll_sampled["traj_generated"] = df_troll_sampled.apply(lambda r: simulate_mixed_policy(r.policy_nt, r.gail_policy, r.tp_nt, r.tp_perturbed_legalised, n, frac=percentage_non_troll), axis=1)
        df_organic_sampled["traj_generated"] = df_organic_sampled.apply(lambda r: simulate_policy(r.gail_policy, r.tp_perturbed_legalised, n), axis=1)

        df_troll_sampled["traj_generated_reshaped"] = df_troll_sampled.apply(data_corruption_utils.reshape_to_n_50_2)
        df_organic_sampled["traj_generated_reshaped"] = df_organic_sampled.apply(data_corruption_utils.reshape_to_n_50_2)

        num_s = 12
        num_a = 6
        def tp(x):
            return irl_utils.legalise_tp(irl_utils.compute_tp(x.reshape(-1, 2), num_s, num_a))
        def state_count(x):
            return irl_utils.compute_state_count(x.reshape(-1, 2), num_s, num_a)
        
        df_troll_sampled["tp_generated"] = df_troll_sampled.apply(tp) 
        df_organic_sampled["tp_generated"] = df_troll_sampled.apply(tp)

        df_troll_sampled["traj_counts_generated"] = df_organic_sampled.apply(state_count)
        df_organic_sampled["traj_counts_generated"] = df_organic_sampled.apply(state_count)

        # # sample embeddings based on generated pol
        df_troll_sampled["embed_generated"] = df_troll_sampled.apply(lambda r: sample_content(r.traj_generated, troll_user_content_df[troll_user_content_df.screen_name == r.user].copy(), r.user), axis=1)
        df_organic_sampled["embed_generated"] = df_organic_sampled.apply(lambda r: sample_content(r.traj_generated, organic_user_content_df[organic_user_content_df.screen_name == r.user].copy(), r.user), axis=1)
        
        troll_generated.append(df_troll_sampled)
        organic_generated.append(df_organic_sampled)

troll_generated_df = pd.concat(troll_generated)
organic_generated_df = pd.concat(organic_generated)


with open(root_output_dir_path + "/df_w_generated_traj_troll_processed.pkl", "wb") as f:
    pickle.dump(troll_generated_df, f)

with open(root_output_dir_path + "/df_w_generated_traj_organic_processed.pkl", "wb") as f:
    pickle.dump(organic_generated_df, f)